In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

matches = pd.read_csv("../data/matches.csv", parse_dates=["GAME_DATE"])

features = ["WIN_PCT_SEASON_HOME", "WIN_PCT_LAST10_HOME", "PM_LAST10_HOME",
            "WIN_PCT_SEASON_AWAY", "WIN_PCT_LAST10_AWAY", "PM_LAST10_AWAY"]
target = "HOME_WIN"

test_seasons = ["2024-25", "2025-26"]
train = matches[~matches["SEASON"].isin(test_seasons)]
test = matches[matches["SEASON"].isin(test_seasons)]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print("Entraînement :", len(train), "matchs | Test :", len(test), "matchs")

Entraînement : 10605 matchs | Test : 2418 matchs


In [2]:
baseline = y_test.mean()
print(f"Baseline (toujours prédire domicile) : {baseline:.1%}")

Baseline (toujours prédire domicile) : 55.0%


In [3]:
model = LogisticRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(f"Modèle : {accuracy_score(y_test, preds):.1%}")

Modèle : 67.0%


In [4]:
weights = pd.Series(model.coef_[0], index=features).sort_values()
weights

WIN_PCT_SEASON_AWAY   -1.872589
WIN_PCT_LAST10_HOME   -0.869134
PM_LAST10_AWAY        -0.036577
PM_LAST10_HOME         0.056785
WIN_PCT_LAST10_AWAY    0.614885
WIN_PCT_SEASON_HOME    2.014441
dtype: float64

In [5]:
X_train[["WIN_PCT_SEASON_HOME", "WIN_PCT_LAST10_HOME", "PM_LAST10_HOME"]].corr().round(2)

,WIN_PCT_SEASON_HOME,WIN_PCT_LAST10_HOME,PM_LAST10_HOME
WIN_PCT_SEASON_HOME,1.00,0.83,0.76
WIN_PCT_LAST10_HOME,0.83,1.00,0.87
PM_LAST10_HOME,0.76,0.87,1.00


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_scaled = make_pipeline(StandardScaler(), LogisticRegression())
model_scaled.fit(X_train, y_train)
print(f"Modèle standardisé : {accuracy_score(y_test, model_scaled.predict(X_test)):.1%}")

pd.Series(model_scaled[-1].coef_[0], index=features).sort_values()

Modèle standardisé : 67.0%


WIN_PCT_SEASON_AWAY   -0.346777
PM_LAST10_AWAY        -0.252244
WIN_PCT_LAST10_HOME   -0.204904
WIN_PCT_LAST10_AWAY    0.153119
PM_LAST10_HOME         0.375102
WIN_PCT_SEASON_HOME    0.382481
dtype: float64

In [7]:
def evaluate(cols):
    """Entraîne un modèle avec les colonnes données et renvoie sa précision sur le test."""
    m = make_pipeline(StandardScaler(), LogisticRegression())
    m.fit(X_train[cols], y_train)
    return accuracy_score(y_test, m.predict(X_test[cols]))

indicators = {
    "Bilan saison":  ["WIN_PCT_SEASON_HOME", "WIN_PCT_SEASON_AWAY"],
    "Forme récente": ["WIN_PCT_LAST10_HOME", "WIN_PCT_LAST10_AWAY"],
    "Domination":    ["PM_LAST10_HOME", "PM_LAST10_AWAY"],
}

print(f"Avec les 6 features      : {evaluate(features):.1%}")
for name, cols in indicators.items():
    remaining = [f for f in features if f not in cols]
    print(f"Sans {name:<14}      : {evaluate(remaining):.1%}")

Avec les 6 features      : 67.0%
Sans Bilan saison        : 64.6%
Sans Forme récente       : 66.1%
Sans Domination          : 65.8%
